# KMD Weather AI: Probabilistic Deep-Learning Forecasting

This notebook builds a complete prediction pipeline for authorized Kenya Meteorological Department WRF data stored outside Git. It will:

1. understand and validate every NetCDF product;
2. convert the meteorology into learning-ready features;
3. train a Temporal-Attention U-Net for **+1 h, +3 h, and +6 h** forecasts;
4. produce p10/p50/p90 uncertainty, rain probability, and full-grid maps;
5. compare the AI with persistence using weather-service verification metrics; and
6. export a reusable checkpoint and a self-describing NetCDF forecast.

> **Scientific honesty gate:** the four supplied files are WRF model forecasts, not observations. Only two files are independent hourly forecast runs. The notebook therefore demonstrates a genuine end-to-end neural forecast, but its scores measure how well the network emulates held-out WRF evolution—not how accurately it predicts observed Kenyan weather. Do not use it for public warnings until it has been retrained and tested against independent observations over multiple seasons.


## 0. Environment

Run the following once in a fresh Jupyter environment, then restart the kernel:

```python
%pip install -r requirements.txt
```


In [ ]:
%pip install -r requirements.txt

In [ ]:
import importlib.util

required = {
    'xarray': 'xarray', 'netCDF4': 'netCDF4', 'torch': 'torch',
    'pandas': 'pandas', 'matplotlib': 'matplotlib', 'tqdm': 'tqdm'
}
missing = [package for package, module in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError(
        f'Missing {missing}. Run `%pip install -r requirements.txt`, restart the kernel, and rerun.'
    )
print('Environment is ready.')


In [ ]:
import os
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xarray as xr
from IPython.display import display
from torch.utils.data import DataLoader

from kmd_forecast.config import get_profile
from kmd_forecast.data import (
    INPUT_FEATURE_NAMES, SpatioTemporalPatchDataset, dataset_inventory,
    fit_normalizer, identify_raw_forecast_files, load_wrf_run, run_summary,
)
from kmd_forecast.evaluation import evaluate_model
from kmd_forecast.inference import export_forecast_netcdf, predict_full_grid
from kmd_forecast.model import MultiTaskQuantileLoss, TemporalAttentionUNet, model_size_megabytes
from kmd_forecast.training import fit_model, save_checkpoint, select_device, set_reproducible
from kmd_forecast.visualization import (
    plot_daily_product, plot_forecast, plot_metric_summary,
    plot_rain_accumulation_check, plot_run_overview, plot_training_history,
)

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold'})

REPOSITORY_DIR = Path.cwd()
DATA_DIR = Path(os.getenv('KMD_DATA_DIR', 'data')).expanduser()
ARTIFACT_DIR = Path(os.getenv('KMD_ARTIFACT_DIR', 'artifacts')).expanduser()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 'demo' runs quickly at half spatial resolution. 'research' uses the native grid.
PROFILE_NAME = 'demo'
config = get_profile(PROFILE_NAME)
set_reproducible(config.seed)
device = select_device()

print(f'Profile: {config.name}')
print(f'Compute device: {device}')
print(f'PyTorch: {torch.__version__}')
display(pd.Series(config.to_dict(), name='value').to_frame())


## 1. Understand the supplied data before modelling

A forecast file is not automatically a training label. We first identify each product, its initialization time, valid-time coverage, and spatial grid. This prevents the daily/weekly forecasts from accidentally being treated as observed truth.


In [ ]:
netcdf_paths = sorted(DATA_DIR.glob('*.nc'))
if not netcdf_paths:
    raise FileNotFoundError(
        f'No NetCDF files found in {DATA_DIR}. See the README data instructions.'
    )

inventory = dataset_inventory(netcdf_paths)
display(inventory)

print('Files:', len(inventory))
print('Independent hourly WRF initializations:', (inventory['product'] == 'hourly WRF run').sum())
print('Domain grid:', inventory.latitude_points.max(), '×', inventory.longitude_points.max())


### What the audit tells us

- `t2` and `d2m` are Kelvin and are converted to °C.
- `msl` is Pa and is converted to hPa.
- `tp` is **accumulated from the start of each WRF run**. It must be differenced before learning hourly rain.
- `u10` and `v10` are wind components; wind speed is derived from them.
- Relative humidity is derived from temperature and dew point.
- Daily and weekly files are useful operational reference products, but they are forecasts from another WRF run—not ground truth.


In [ ]:
raw_paths = identify_raw_forecast_files(netcdf_paths)
if len(raw_paths) < 2:
    raise ValueError('At least two independent hourly WRF runs are needed for a run-level split.')

print('Loading and engineering hourly runs...')
runs = [load_wrf_run(path, spatial_stride=config.spatial_stride) for path in raw_paths]
display(run_summary(runs))
print('Input channels:', len(INPUT_FEATURE_NAMES), INPUT_FEATURE_NAMES)


In [ ]:
plot_run_overview(runs[-1], time_index=-1)
plt.show()

plot_rain_accumulation_check(raw_paths[-1])
plt.show()

daily_paths = [path for path in netcdf_paths if 'daily' in path.name.lower()]
if daily_paths:
    plot_daily_product(daily_paths[0], day_index=0)
    plt.show()


## 2. Leakage-safe experiment design

Weather grids next to one another are highly correlated. A random pixel split would give unrealistically good scores. Instead, the earlier WRF initialization is training data and the newest initialization is held out in full for validation. Normalization is fitted only on training data.

Each sample uses the previous 6 hourly grids and directly predicts +1 h, +3 h, and +6 h. Direct multi-horizon prediction avoids recursive error growth. Spatial patches create many learning examples while retaining real weather structures.


In [ ]:
training_runs = runs[:-1]
validation_runs = runs[-1:]
normalizer = fit_normalizer(training_runs, config.rain_threshold_mm)

training_dataset = SpatioTemporalPatchDataset(
    training_runs, normalizer, config.history_hours, config.horizons,
    config.patch_size, config.train_patches_per_anchor,
    config.rain_threshold_mm, training=True, seed=config.seed,
)
validation_dataset = SpatioTemporalPatchDataset(
    validation_runs, normalizer, config.history_hours, config.horizons,
    config.patch_size, config.validation_patches_per_anchor,
    config.rain_threshold_mm, training=False, seed=config.seed,
)

loader_options = dict(
    batch_size=config.batch_size, num_workers=config.num_workers,
    pin_memory=(device.type == 'cuda'),
)
training_loader = DataLoader(training_dataset, shuffle=True, **loader_options)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)

print('Training run(s):', [run.path.name for run in training_runs])
print('Validation run:', [run.path.name for run in validation_runs])
print('Training patches per epoch:', len(training_dataset))
print('Validation patches:', len(validation_dataset))
print('Rain positive-class weight:', round(normalizer.wet_positive_weight, 2))


## 3. High-tech model, explained simply

The model first applies 3-D convolutions across **time and space**. Learned temporal attention decides which of the past six hours matters at each location. A U-Net captures local-to-regional weather structures, and global self-attention at its compressed bottleneck connects distant regions. Three heads produce temperature quantiles, rainfall quantiles, and the probability of measurable rain.

The loss combines quantile regression with a weighted wet/dry classification loss. This is more appropriate than plain mean-squared error for sparse, highly skewed rainfall.


In [ ]:
model = TemporalAttentionUNet(
    input_channels=len(INPUT_FEATURE_NAMES),
    horizons=config.horizons,
    quantiles=config.quantiles,
    base_channels=config.base_channels,
    dropout=config.dropout,
)
criterion = MultiTaskQuantileLoss(
    quantiles=config.quantiles,
    wet_positive_weight=normalizer.wet_positive_weight,
).to(device)

print(f'Trainable parameters: {model.parameter_count:,}')
print(f'Parameter storage: {model_size_megabytes(model):.1f} MB')
sample = next(iter(training_loader))
with torch.no_grad():
    sample_output = model(sample['x'][:1])
print({name: tuple(value.shape) for name, value in sample_output.items()})


## 4. Train

Training uses AdamW, one-cycle learning rates, gradient clipping, mixed precision on NVIDIA GPUs, early stopping, and a deterministic seed. The `demo` profile intentionally trains only a few epochs so the complete notebook is practical; use `research` after adding a much larger historical archive.


In [ ]:
history = fit_model(
    model=model,
    train_loader=training_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    device=device,
    epochs=config.epochs,
    patience=config.patience,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
)
display(history.round(4))
plot_training_history(history)
plt.show()


## 5. Verify against a baseline

A complex model is useful only if it beats a simple forecast. Persistence says the future equals the latest input. We report MAE, RMSE, and bias for continuous fields; rain also receives Brier score (probability quality), probability of detection (POD), false-alarm ratio (FAR), and critical success index (CSI). Positive MAE skill means the AI beats persistence.


In [ ]:
metrics = evaluate_model(
    model, validation_loader, normalizer, config.horizons, config.quantiles,
    config.rain_threshold_mm, device,
)
metrics_path = ARTIFACT_DIR / 'validation_metrics.csv'
metrics.to_csv(metrics_path, index=False)
display(metrics.round(4))
plot_metric_summary(metrics)
plt.show()
print('Saved:', metrics_path)


### How to read these scores

- A negative AI bias means systematic underprediction; a positive bias means overprediction.
- Lower Brier score is better. CSI balances missed rain and false alarms; higher is better.
- p10–p90 coverage should be near 0.80 on a large independent test set.
- With the current tiny archive, results can vary strongly between weather systems. This is a pipeline acceptance test, not an operational model-selection result.


## 6. Make an actual future prediction

The latest six frames of the newest WRF run initialize the neural model. Overlapping tiles control memory and are smoothly blended into one complete-domain probabilistic forecast. These valid times extend beyond the last hourly frame supplied to the network.


In [ ]:
latest_run = runs[-1]
forecast = predict_full_grid(
    model=model, run=latest_run, normalizer=normalizer,
    history_hours=config.history_hours, horizons=config.horizons,
    quantiles=config.quantiles, tile_size=config.patch_size,
    overlap=config.tile_overlap, device=device, tile_batch_size=4,
)
plot_forecast(forecast)
plt.show()


### Easy-to-read forecast for Nairobi

The table below extracts the nearest grid cell. The spread from p10 to p90 represents model uncertainty learned from the training objective; it is not yet calibrated against observations.


In [ ]:
NAIROBI_LAT, NAIROBI_LON = -1.2864, 36.8172
iy = int(np.argmin(np.abs(forecast.latitude - NAIROBI_LAT)))
ix = int(np.argmin(np.abs(forecast.longitude - NAIROBI_LON)))
q10, q50, q90 = 0, int(np.argmin(np.abs(forecast.quantiles - 0.5))), -1

point_rows = []
for index, horizon in enumerate(forecast.horizons):
    probability = float(forecast.wet_probability[index, iy, ix])
    rain_p50 = float(forecast.rain_mm[index, q50, iy, ix]) if probability >= 0.5 else 0.0
    point_rows.append({
        'valid_time_utc': pd.Timestamp(forecast.valid_times[index]),
        'lead_hours': int(horizon),
        'temperature_p10_c': forecast.temperature_c[index, q10, iy, ix],
        'temperature_p50_c': forecast.temperature_c[index, q50, iy, ix],
        'temperature_p90_c': forecast.temperature_c[index, q90, iy, ix],
        'rain_probability_pct': probability * 100,
        'operational_rain_p50_mm': rain_p50,
        'rain_p90_mm': forecast.rain_mm[index, q90, iy, ix],
    })
point_forecast = pd.DataFrame(point_rows)
display(point_forecast.round(2))
print(f'Nearest grid point: {forecast.latitude[iy]:.3f}, {forecast.longitude[ix]:.3f}')


## 7. Export deployable artifacts

The checkpoint contains weights, normalization, feature order, configuration, and learning history. The NetCDF contains coordinates, valid times, forecast periods, quantiles, units, and a research-use disclaimer.


In [ ]:
checkpoint_path = save_checkpoint(
    ARTIFACT_DIR / 'kmd_temporal_attention_unet.pt',
    model, normalizer, config.to_dict(), history,
    extra={
        'training_files': [run.path.name for run in training_runs],
        'validation_files': [run.path.name for run in validation_runs],
        'scientific_status': 'proof-of-pipeline; requires observation-based validation',
    },
)
forecast_path = export_forecast_netcdf(
    forecast, ARTIFACT_DIR / 'kmd_ai_forecast.nc'
)
print('Checkpoint:', checkpoint_path)
print('Forecast:', forecast_path)

with xr.open_dataset(forecast_path) as exported:
    display(exported)


## 8. Path from prototype to a real KMD operational service

The software pipeline is complete. Scientific readiness now depends on data and governance:

1. **Build a historical archive:** ideally 2–5 years of identical WRF initialization cycles and lead times, including long rains, short rains, dry seasons, and extremes.
2. **Use measured targets:** quality-controlled KMD stations for temperature/wind; gauge-corrected radar, CHIRPS, IMERG, or another approved analysis for rain. Keep station QC flags and missingness masks.
3. **Align without leakage:** join by valid time and location. Split by whole months/years and use rolling-origin backtests. Never let later forecast runs enter earlier training windows.
4. **Calibrate by region and season:** report skill for counties, elevations, lead times, rain thresholds, and seasons—not only one national average. Calibrate p10/p90 and rain probabilities.
5. **Add static geography:** elevation, land/sea mask, slope, land use, and optionally soil moisture. These are especially valuable over Kenya's terrain.
6. **Operational safeguards:** schema checks, missing-input fallback to WRF, drift monitoring, latency alarms, model registry, reproducible versioning, and human forecaster approval.
7. **Go/no-go rule:** promote only if the model beats raw WRF and persistence on an untouched observation-based test year, improves high-impact rain CSI without unacceptable false alarms, and has calibrated uncertainty.

A strong next version should learn **corrections to WRF against observations** (model-output statistics) rather than merely emulating WRF. That preserves the physics of the numerical model while letting deep learning correct local Kenyan bias and resolution errors.
